# Route 97 recurring segment delay prediction

Can recurring calendar patterns predict how much delay a bus **gains or recovers on a directed stop-to-stop segment**? This separate experiment builds on [the delay-congestion analysis](route_97_delay_congestion_and_propagation.ipynb), without changing it.

**Target (seconds):** downstream final reported arrival delay minus upstream final reported arrival delay. Positive means added delay; negative means recovery. Each completed traversal contributes once. Missing observations are never zero-filled.

We compare **absolute-error XGBoost** with ordinary and robust calendar averages, plus zero-change and segment-only references. The goal is typical repeatable patterns, **not unexpected incidents**. No anomaly labels exist: our statistical tail flags are diagnostics, not verified event classifications.

**Limits:** these are final-reported proxies, not measured arrivals. Arrival-to-arrival delay change includes dwell, holding, schedule slack and prediction revisions; it is not pure road traffic. Out-of-time predictability does not establish causality or improvement to an operational arrival estimator.

## Environment and reproducibility

Use the `transit_env` kernel. Install declared dependencies with `conda run -n transit_env python -m pip install -e ./forecasting` from the repository root if needed. The experiment is **offline**: it loads the explicitly pinned trusted snapshot and its static GTFS, never credentials or replacement data. Pickle files must come from trusted sources.

Reusable transformations, splits, fitting, scoring and persistence live in `forecasting.segment_delay_model` and have focused pytest coverage. The bootstrap below also supports launching this notebook from the repository root.

In [ ]:
import json
import pickle
import sys
from pathlib import Path

repo_root = next(p for p in (Path.cwd(), Path.cwd().parent) if (p / 'forecasting' / 'forecasting').is_dir())
sys.path.insert(0, str(repo_root / 'forecasting'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from IPython.display import Markdown, display
from forecasting import (
    DelayModelConfig, get_backtest_dataset, make_segment_delay_splits,
    prepare_segment_delay_data, run_delay_congestion_engine,
    run_segment_delay_experiment, save_segment_delay_experiment,
    summarize_segment_delay_uplift,
)

if Path(sys.prefix).name != 'transit_env':
    raise RuntimeError(f'Select the transit_env kernel, not {sys.executable}')
pio.renderers.default = 'plotly_mimetype'
pd.options.display.float_format = '{:,.3f}'.format
print(f'Python: {sys.executable}')

In [ ]:
dataset_id = '2026-11-10_2_months'
route_id = '97-KEL'
direction_id = 0
agency_timezone = 'America/Vancouver'
time_bin_minutes = 30
config = DelayModelConfig(n_splits=2, block_days=7, gap_days=1, min_train_days=21)
print(config)
print('All metrics and predictions use seconds; signed recovery values are retained.')

## Load and audit the historical evidence

Reusing the existing engine preserves exact trip-instance pairing, adjacent scheduled stops, shape/direction identity and label availability. Latest missing estimates remain missing; excluded endpoints are never bridged. This snapshot has no receipt timestamps, so availability is publication-time replay, not verified receipt-time replay.

The source spans September 11–November 10, but many early trips do not match its saved static schedule. **Four** weekly CV blocks would leave only about nine observed initial training days. The approved adjustment is **two weekly blocks**, retaining at least 21 observed training days and the final weekly test. Coverage and exclusions below remain part of the evidence; this is not a two-month fully matched sample.

In [ ]:
dataset = get_backtest_dataset(repo_root / 'data', source='cache', dataset_id=dataset_id)
manifest = json.loads((dataset.directory / 'metadata.json').read_text())
id_dtypes = {name: 'string' for name in ('trip_id', 'stop_id', 'shape_id', 'route_id')}
tables = {name: pd.read_csv(dataset.static_data_dir / f'{name}.txt', dtype=id_dtypes)
          for name in ('stops', 'stop_times', 'trips', 'shapes')}
delay_result = run_delay_congestion_engine(
    dataset.trip_updates, tables['stops'], tables['stop_times'], tables['trips'], tables['shapes'],
    route_id=route_id, trip_direction_id=direction_id,
)
modelling_data = prepare_segment_delay_data(
    delay_result, tables['stop_times'], observation_start=dataset.trip_updates.feed_timestamp.min(),
    timezone=agency_timezone, time_bin_minutes=time_bin_minutes,
)
examples = modelling_data.examples
display(modelling_data.audit)
print(f'{len(examples):,} usable complete-day traversals; {examples.segment_id.nunique()} observed segments')

In [ ]:
display(modelling_data.coverage)
segment_support = examples.groupby('segment_id').agg(traversals=('delta_delay_seconds', 'size'), days=('service_date', 'nunique'))
display(modelling_data.segments.merge(segment_support, on='segment_id', how='left'))
px.bar(modelling_data.coverage, x='service_date', y='traversals',
       title='Usable service-day coverage (missing days remain visible)').show()

## Advance features and blocked time-series cross-validation

Features are **segment identity, local weekday, scheduled upstream arrival minutes**, and daily/weekly sine/cosine encodings. Each traversal uses its **own** scheduled stop sequence, not a representative trip. GTFS times beyond 24:00 and daylight saving are handled using service dates and the GTFS noon-minus-12-hours origin. Integer segment IDs are categorical, not distances; the saved mapping identifies their directed shape-specific stop pairs.

**No actual crossing time, endpoint delay, trip ID, full-history anomaly estimate or realtime observation enters the predictor.** Ordinary averages use `(segment, weekday, 30-minute clock bin)`; XGBoost can learn smoother time interactions from the same clock/calendar information.

Each validation block is a consecutive **calendar week**, with expanding past-only training. A full service-day gap precedes it. Whole trips/service days stay together; training trips with any label unavailable at the fit cutoff are purged. Recording-edge partial service days are excluded. The last complete seven-day block is reserved before tuning. Missing evaluation days or insufficient initial history raise explicit errors rather than silently shrinking the design.

In [ ]:
splits, split_table = make_segment_delay_splits(modelling_data, config)
display(split_table)
timeline = pd.concat([
    split_table[['split', 'train_start', 'train_end']].rename(columns={'train_start': 'start', 'train_end': 'end'}).assign(role='training', end=lambda t: t.end + pd.Timedelta(days=1)),
    split_table[['split', 'evaluation_start', 'evaluation_end_exclusive']].rename(columns={'evaluation_start': 'start', 'evaluation_end_exclusive': 'end'}).assign(role='held out'),
])
px.timeline(timeline, x_start='start', x_end='end', y='split', color='role',
            title='Past-only blocks: blank interval is the training gap').show()

## Learn repeatable behaviour without chasing incident magnitudes

- **XGBoost:** absolute-error (`reg:absoluteerror`) loss on original signed outcomes. L1 loss reduces the influence of isolated extreme magnitudes without dropping every large rush-hour observation. A small fixed grid controls tree complexity; no held-out labels are used for early stopping.
- **Required baseline:** ordinary arithmetic averages of the segment/weekday/30-minute cohorts.
- **Robust average:** arithmetic averages after training-only winsorisation around a conditional median reference. This checks whether XGBoost merely benefits from an outlier-resistant objective.
- **References:** zero change and robust segment-only averages. Calendar models must improve beyond fixed segment corrections to support the recurring-calendar hypothesis.

Calendar tables require at least **five observations across three service days**; fallback is segment/time bin, segment, then global training location. Fallback levels, sparse/unseen segments and zero-width residual bands are explicit diagnostics, not hidden defaults.

**Typical-range diagnostics:** within each training partition, fit calendar-conditioned medians and obtain leave-one-training-service-day-out residuals. Use each supported segment's 95th percentile of absolute residuals as its band, with an explicit global training-only fallback. The validation/test mask compares its original outcome to the frozen training reference/band. This conditional reference protects repeatable peaks better than trimming globally large delay changes. Repeat at 99% as a predeclared sensitivity check.

These masks are retrospective evaluation cohorts, **not deployable incident detection**. An unusual outcome can be a rare genuine recurring pattern; drift can increase the tail share. All models share the same mask. **Held-out outcomes are never removed from the all-observation scores or clipped.** Tail errors and coverage are shown separately.

Select XGBoost by mean CV **typical-range macro-segment MAE** at 95%. Display all-observation metrics alongside it. Selected CV scores are tuning estimates; the untouched final week is the confirmation, not another tuning set.

In [ ]:
experiment = run_segment_delay_experiment(modelling_data, config)
display(experiment.tuning)
display(experiment.tuning.groupby('candidate')[['selection_mae_seconds', 'all_mae_seconds', 'all_rmse_seconds', 'typical_coverage']].mean())
print(f'Selected using development blocks only: {experiment.selected_candidate}')
display(experiment.band_diagnostics)
print('Zero-width bands:', int(experiment.band_diagnostics.degenerate_band.sum()))

## Regression accuracy: typical behaviour, full outcomes, and tails

**MAE** measures typical absolute error in seconds; **median AE** resists a few large errors. **RMSE** deliberately exposes sensitivity to large misses; **R²** provides variance-relative context and may be negative. Constant or tiny cohorts have undefined R² with an explicit reason. **Macro-segment MAE** weights each observed segment equally; ordinary MAE weights traversals equally.

No MAPE (zero/negative targets) or log metrics. Coverage, row counts and segment counts accompany every score. Results are reported on unchanged targets for all observations, the common typical cohort, tails, and known/cold-start segments. All-observation metrics must not be interpreted as the objective of predicting incidents.

In [ ]:
metrics = experiment.metrics
score_columns = ['split', 'cohort', 'model', 'n', 'coverage', 'segments', 'mae_seconds',
                 'median_ae_seconds', 'rmse_seconds', 'r2', 'macro_segment_mae_seconds', 'r2_undefined_reason']
display(metrics.loc[metrics.cohort.isin(['all', 'typical_q0.95']), score_columns])
display(metrics.loc[metrics.phase.eq('test'), score_columns])
plot_scores = metrics.loc[metrics.cohort.eq('typical_q0.95') & metrics.model.isin([
    'zero', 'segment_mean_q0.95', 'calendar_mean', 'robust_mean_q0.95', 'xgboost'])]
px.bar(plot_scores, x='split', y='macro_segment_mae_seconds', color='model', barmode='group',
       title='Equal-segment typical-range MAE by held-out week (seconds)').show()

In [ ]:
display(metrics.loc[metrics.cohort.isin(['typical_q0.95', 'tail_q0.95', 'typical_q0.99', 'tail_q0.99']) &
                    metrics.model.isin(['calendar_mean', 'robust_mean_q0.95', 'robust_mean_q0.99', 'xgboost']), score_columns])
predictions = experiment.predictions
support_columns = [c for c in predictions if c.endswith('_level')]
fallback_counts = predictions.groupby('split')[support_columns].agg(lambda values: values.value_counts().to_dict())
display(fallback_counts)
display(predictions.groupby('split').agg(rows=('cold_start', 'size'), cold_start_rows=('cold_start', 'sum'),
                                        typical_95_share=('typical_q0.95', 'mean'), typical_99_share=('typical_q0.99', 'mean')))

## Is there evidence of repeated calendar patterns?

Positive paired improvement means XGBoost has lower MAE. Compare with **both calendar averages** and the **segment-only** reference. Inspect consistency across weeks, per-segment support and the 99% sensitivity band; a pooled win on one busy segment is insufficient. Segment rows from the same bus/day are correlated, so the daily table is the unit of descriptive consistency, not thousands of independent trials.

The two CV weeks and one test week limit conclusions. Feature importance or a single positive score is not proof of stable future benefit. These profiles condition on predictable calendar inputs, not actual traffic incidents or causal effects.

In [ ]:
segment_uplift, daily_uplift = summarize_segment_delay_uplift(predictions)
display(daily_uplift)
display(segment_uplift.loc[segment_uplift.phase.eq('test')].merge(modelling_data.segments[
    ['segment_id', 'from_stop_name', 'to_stop_name']], on='segment_id'))
px.bar(segment_uplift, x='segment_id', y='improvement_vs_robust_mean_q0.95_seconds', color='split',
       barmode='group', hover_data=['n', 'service_days'],
       title='Paired typical MAE improvement over robust calendar averages (positive is better)').show()

In [ ]:
final = predictions.loc[predictions.phase.eq('test')].copy()
final['local_hour'] = final.scheduled_at.dt.tz_convert(agency_timezone).dt.hour
profile_columns = ['delta_delay_seconds', 'xgboost', 'calendar_mean', 'robust_mean_q0.95']
profiles = final.loc[final['typical_q0.95']].groupby(['segment_id', 'local_hour'])[profile_columns].mean().reset_index()
px.line(profiles.melt(id_vars=['segment_id', 'local_hour'], var_name='series', value_name='seconds'),
        x='local_hour', y='seconds', color='series', facet_col='segment_id', facet_col_wrap=4,
        title='Final-week typical profiles: original outcomes versus advance calendar predictions', height=1100).show()
trace_segment = final.segment_id.value_counts().sort_index().idxmax()
trace = final.loc[final.segment_id.eq(trace_segment)].sort_values('scheduled_at')
figure = px.line(trace.melt(id_vars=['scheduled_at'], value_vars=profile_columns, var_name='series', value_name='seconds'),
                 x='scheduled_at', y='seconds', color='series', title=f'Final-week chronological trace, segment {trace_segment}')
tail = trace.loc[~trace['typical_q0.95']]
figure.add_scatter(x=tail.scheduled_at, y=tail.delta_delay_seconds, mode='markers', name='statistical tail (unchanged target)', marker={'symbol': 'x', 'size': 10})
figure.show()

In [ ]:
final_scores = metrics.loc[metrics.phase.eq('test') & metrics.cohort.eq('typical_q0.95')].set_index('model')
xgb_mae = final_scores.loc['xgboost', 'mae_seconds']
comparisons = final_scores.loc[['calendar_mean', 'robust_mean_q0.95', 'segment_mean_q0.95'], ['mae_seconds', 'macro_segment_mae_seconds']].copy()
comparisons['xgboost_improvement_seconds'] = comparisons.mae_seconds - xgb_mae
display(comparisons)
daily_test = daily_uplift.loc[daily_uplift.phase.eq('test')]
positive_days = int(daily_test['improvement_vs_robust_mean_q0.95_seconds'].gt(0).sum())
weekly = metrics.loc[metrics.cohort.eq('typical_q0.95')].pivot(index='split', columns='model', values='mae_seconds')
calendar_wins = int(weekly['robust_mean_q0.95'].lt(weekly['segment_mean_q0.95']).sum())
increment = final_scores.loc['robust_mean_q0.95', 'mae_seconds'] - xgb_mae
display(Markdown(f'### Experiment conclusion\n\n**Recurring calendar signal:** robust calendar averages beat the fixed segment-only correction in **{calendar_wins}/{len(weekly)} held-out weeks**. This supports investigating repeated time-of-day/weekday patterns in these matched segments, not pure road-traffic causality.\n\n**Incremental XGBoost value:** final-week typical MAE is **{xgb_mae:.2f} seconds**, a **{increment:.2f}-second** pooled improvement over robust calendar averages. XGBoost wins on **{positive_days}/{len(daily_test)} held-out days**. A small pooled gain with mixed daily results does not establish a reliable advantage over the simpler robust baseline. Inspect macro-segment MAE, RMSE and the 99% band as well; no operational uplift is proven.\n\nThese few held-out weeks are a feasibility check, not a significance claim. Additional seasons, matched historical schedules, labelled disruptions and an operational upstream-information backtest are needed before integration.'))

## Save the evidence and fitted pre-test model

The run below uses a new directory under ignored `data/segment_delay_models/` and refuses to overwrite a run. It stores the model/preprocessing and baseline/threshold state, snapshot checksums, configuration and versions, segment mapping, split audit, tuning results, untouched out-of-time predictions, metric tables and paired uplift. Notebook tables and figures remain embedded here.

The saved model was trained **before the final test** with the same gap and availability rules. It was not subsequently refit on test outcomes. Its `predict` method needs only prepared calendar features, not observed outcomes or an anomaly flag. Reload **trusted** bundles only. Preparing arbitrary future timetable requests and wiring them into the live estimator are outside this experiment.

In [ ]:
run_directory = save_segment_delay_experiment(
    experiment, modelling_data, repo_root / 'data' / 'segment_delay_models', dataset_metadata=manifest,
)
with (run_directory / 'model_bundle.pkl').open('rb') as stream:
    reloaded_model = pickle.load(stream)
test_examples = examples.loc[splits[-1].evaluation_index]
reloaded_predictions = reloaded_model.predict(test_examples.drop(columns='delta_delay_seconds'))
np.testing.assert_array_equal(reloaded_predictions.xgboost.to_numpy(), final.xgboost.to_numpy())
print(f'Saved: {run_directory}')
print('Verified: reloaded model reproduces every final-test XGBoost prediction exactly.')